# startup-investor recommendation engine

## 1. imports

In [2]:
import pandas as pd
import numpy as np
import time
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer

np.random.seed(42)
print("imports done")

imports done


## 2. load data from csv

In [3]:
startups_df  = pd.read_csv("startups.csv")
investors_df = pd.read_csv("investors.csv")

print(f"startups  loaded : {len(startups_df)} rows, {list(startups_df.columns)}")
print(f"investors loaded : {len(investors_df)} rows, {list(investors_df.columns)}")

startups  loaded : 45 rows, ['id', 'name', 'sector', 'stage', 'location', 'funding_needed', 'traction', 'team_score', 'market_size', 'founded_year', 'revenue_arr', 'employees', 'website', 'business_model', 'problem', 'solution', 'description', 'tags']
investors loaded : 10 rows, ['id', 'name', 'firm', 'preferred_sectors', 'preferred_stages', 'min_investment', 'max_investment', 'location', 'portfolio_count', 'focus', 'notable_exits']


## 3. preview data

In [4]:
print("--- startups sample ---")
print(startups_df[["id", "name", "sector", "stage", "revenue_arr", "traction", "team_score"]].to_string(index=False))

--- startups sample ---
         id           name     sector    stage  revenue_arr  traction  team_score
startup_001        PayNova    FinTech Series A         2.40       8.7         9.1
startup_002     WealthPath    FinTech     Seed         0.60       7.2         8.3
startup_003       InsureIQ    FinTech Series B         7.80       9.2         9.4
startup_004       LoanDost    FinTech Pre-Seed         0.10       6.1         7.8
startup_005      TradeNest    FinTech Series A         1.90       8.1         8.6
startup_006    MediScan AI HealthTech Series A         3.10       8.9         9.3
startup_007     CareCircle HealthTech     Seed         0.50       7.4         8.0
startup_008       NutriGen HealthTech Pre-Seed         0.08       5.8         7.5
startup_009         SurgAI HealthTech Series B         9.20       9.0         9.6
startup_010      MindBloom HealthTech     Seed         0.70       7.6         8.2
startup_011     SkillForge     EdTech Series A         2.80       8.4     

In [5]:
print("--- investors sample ---")
print(investors_df[["id", "name", "firm", "preferred_sectors", "preferred_stages"]].to_string(index=False))

--- investors sample ---
     id          name                  firm           preferred_sectors       preferred_stages
inv_001 Ananya Kapoor   Inflection Ventures       FinTech|SaaS|DeepTech          Seed|Series A
inv_002   Rajan Mehta    BioHorizon Capital BioTech|HealthTech|DeepTech      Series A|Series B
inv_003    Priya Nair      GreenFuture Fund     CleanTech|AgriTech|SaaS Pre-Seed|Seed|Series A
inv_004  Vikram Sinha       GameOn Ventures    GameTech|E-Commerce|SaaS          Seed|Series A
inv_005 Sneha Agarwal      EduQuest Capital      EdTech|SaaS|HealthTech Pre-Seed|Seed|Series A
inv_006  Rahul Tiwari      DeepTech Foundry       DeepTech|FinTech|SaaS Seed|Series A|Series B
inv_007   Meera Joshi   AgriGrowth Partners  AgriTech|CleanTech|FinTech          Pre-Seed|Seed
inv_008    Arjun Bose ConsumerFirst Capital  E-Commerce|GameTech|EdTech      Series A|Series B
inv_009   Deepika Rao  HealthVentures India HealthTech|BioTech|DeepTech Seed|Series A|Series B
inv_010  Suresh Kumar    

## 4. data models

In [6]:
@dataclass
class Startup:
    id: str
    name: str
    sector: str
    stage: str
    location: str
    funding_needed: float
    traction: float
    team_score: float
    market_size: float
    founded_year: int
    revenue_arr: float
    employees: int
    website: str
    business_model: str
    problem: str
    solution: str
    description: str
    tags: List[str] = field(default_factory=list)


@dataclass
class Investor:
    id: str
    name: str
    firm: str
    preferred_sectors: List[str]
    preferred_stages: List[str]
    min_investment: float
    max_investment: float
    location: str
    portfolio_count: int
    focus: str
    notable_exits: List[str] = field(default_factory=list)


@dataclass
class InteractionEvent:
    investor_id: str
    startup_id: str
    event_type: str
    weight: float
    timestamp: float

print("data models defined")

data models defined


## 5. parse csv into dataclass objects

In [7]:
def parse_startups(df: pd.DataFrame) -> List[Startup]:
    startups = []
    for _, row in df.iterrows():
        tags = str(row["tags"]).split("|") if pd.notna(row["tags"]) else []
        startups.append(Startup(
            id=row["id"],
            name=row["name"],
            sector=row["sector"],
            stage=row["stage"],
            location=row["location"],
            funding_needed=float(row["funding_needed"]),
            traction=float(row["traction"]),
            team_score=float(row["team_score"]),
            market_size=float(row["market_size"]),
            founded_year=int(row["founded_year"]),
            revenue_arr=float(row["revenue_arr"]),
            employees=int(row["employees"]),
            website=str(row["website"]),
            business_model=str(row["business_model"]),
            problem=str(row["problem"]),
            solution=str(row["solution"]),
            description=str(row["description"]),
            tags=tags,
        ))
    return startups


def parse_investors(df: pd.DataFrame) -> List[Investor]:
    investors = []
    for _, row in df.iterrows():
        preferred_sectors = str(row["preferred_sectors"]).split("|")
        preferred_stages  = str(row["preferred_stages"]).split("|")
        notable_exits     = str(row["notable_exits"]).split("|") if pd.notna(row["notable_exits"]) else []
        investors.append(Investor(
            id=row["id"],
            name=row["name"],
            firm=row["firm"],
            preferred_sectors=preferred_sectors,
            preferred_stages=preferred_stages,
            min_investment=float(row["min_investment"]),
            max_investment=float(row["max_investment"]),
            location=row["location"],
            portfolio_count=int(row["portfolio_count"]),
            focus=str(row["focus"]),
            notable_exits=notable_exits,
        ))
    return investors


STARTUPS  = parse_startups(startups_df)
INVESTORS = parse_investors(investors_df)

print(f"parsed {len(STARTUPS)} startups")
print(f"parsed {len(INVESTORS)} investors")

parsed 45 startups
parsed 10 investors


## 6. feature builder

In [8]:
class FeatureBuilder:
    SECTOR_LIST = ["FinTech", "HealthTech", "EdTech", "CleanTech", "AgriTech",
                   "DeepTech", "SaaS", "E-Commerce", "GameTech", "BioTech"]
    STAGE_LIST  = ["Pre-Seed", "Seed", "Series A", "Series B", "Series C"]

    def __init__(self, startups: List[Startup]):
        self.startups = startups
        self.scaler   = MinMaxScaler()
        self.tfidf    = TfidfVectorizer(max_features=40, stop_words="english")
        self._build()

    def _build(self):
        rows, texts = [], []
        for s in self.startups:
            sector_vec = [1 if s.sector == sec else 0 for sec in self.SECTOR_LIST]
            stage_vec  = [1 if s.stage  == stg else 0 for stg in self.STAGE_LIST]
            numeric    = [s.funding_needed, s.traction, s.team_score,
                          s.market_size, s.revenue_arr, s.employees]
            rows.append(sector_vec + stage_vec + numeric)
            texts.append(
                s.description + " " + s.problem + " " + s.solution + " " + " ".join(s.tags)
            )
        mat = np.array(rows, dtype=float)
        mat[:, -6:]         = self.scaler.fit_transform(mat[:, -6:])
        tfidf_mat           = self.tfidf.fit_transform(texts).toarray()
        self.feature_matrix = np.hstack([mat, tfidf_mat])
        self.id_to_idx      = {s.id: i for i, s in enumerate(self.startups)}
        self.idx_to_id      = {i: s.id for i, s in enumerate(self.startups)}

    def content_similarity_matrix(self) -> np.ndarray:
        return cosine_similarity(self.feature_matrix)

print("feature builder defined")

feature builder defined


## 7. interaction tracker

In [9]:
class InteractionTracker:
    WEIGHTS = {"fund": 5.0, "save": 3.0, "click": 1.0, "skip": -1.5}

    def __init__(self):
        self.history         = defaultdict(lambda: deque(maxlen=500))
        self.sector_affinity = defaultdict(lambda: defaultdict(float))
        self.stage_affinity  = defaultdict(lambda: defaultdict(float))

    def record(self, investor_id: str, startup: Startup, event_type: str):
        weight = self.WEIGHTS.get(event_type, 0.0)
        event  = InteractionEvent(
            investor_id=investor_id,
            startup_id=startup.id,
            event_type=event_type,
            weight=weight,
            timestamp=time.time(),
        )
        self.history[investor_id].append(event)
        self.sector_affinity[investor_id][startup.sector] += weight
        self.stage_affinity[investor_id][startup.stage]   += weight

    def get_sector_weights(self, investor_id: str) -> Dict[str, float]:
        raw   = dict(self.sector_affinity[investor_id])
        if not raw:
            return {}
        total = sum(abs(v) for v in raw.values()) or 1.0
        return {k: v / total for k, v in raw.items()}

    def get_seen_ids(self, investor_id: str) -> set:
        return {e.startup_id for e in self.history[investor_id]}

    def interaction_vector(self, investor_id: str, startup_ids: List[str]) -> np.ndarray:
        id_set = {e.startup_id: e.weight for e in self.history[investor_id]}
        return np.array([id_set.get(sid, 0.0) for sid in startup_ids])

print("interaction tracker defined")

interaction tracker defined


## 8. recommendation engine

In [10]:
class RecommendationEngine:
    CB_WEIGHT   = 0.40
    CF_WEIGHT   = 0.35
    PREF_WEIGHT = 0.25

    def __init__(self, startups: List[Startup], investors: List[Investor]):
        self.startups     = startups
        self.investors    = investors
        self.startup_map  = {s.id: s for s in startups}
        self.investor_map = {i.id: i for i in investors}
        self.startup_ids  = [s.id for s in startups]
        print("building feature matrix...")
        self.features    = FeatureBuilder(startups)
        self.content_sim = self.features.content_similarity_matrix()
        self.tracker     = InteractionTracker()
        print(f"engine ready — {len(startups)} startups, {len(investors)} investors")

    def record_interaction(self, investor_id: str, startup_id: str, event_type: str):
        self.tracker.record(investor_id, self.startup_map[startup_id], event_type)

    def recommend(
        self,
        investor_id: str,
        seed_startup_id: Optional[str] = None,
        n: int = 10,
        exclude_seen: bool = True,
    ) -> List[Tuple[Startup, float, str]]:
        seen_ids   = self.tracker.get_seen_ids(investor_id) if exclude_seen else set()
        candidates = [sid for sid in self.startup_ids
                      if sid not in seen_ids and sid != seed_startup_id]
        scores: Dict[str, Dict[str, float]] = {}

        if seed_startup_id and seed_startup_id in self.features.id_to_idx:
            seed_idx = self.features.id_to_idx[seed_startup_id]
            for sid in candidates:
                idx = self.features.id_to_idx[sid]
                scores[sid] = {"cb": float(self.content_sim[seed_idx][idx]), "cf": 0.0, "pref": 0.0}
        else:
            for sid in candidates:
                scores[sid] = {"cb": 0.0, "cf": 0.0, "pref": 0.0}

        cf_scores = self._collaborative_scores(investor_id, candidates)
        for sid in candidates:
            scores[sid]["cf"] = cf_scores.get(sid, 0.0)

        sector_w = self.tracker.get_sector_weights(investor_id)
        investor = self.investor_map.get(investor_id)
        for sid in candidates:
            s    = self.startup_map[sid]
            pref = sector_w.get(s.sector, 0.0)
            if investor and s.sector in investor.preferred_sectors:
                pref += 0.10
            if investor and s.stage in investor.preferred_stages:
                pref += 0.05
            scores[sid]["pref"] = min(pref, 1.0)

        final = []
        for sid, s in scores.items():
            score  = (self.CB_WEIGHT * s["cb"] +
                      self.CF_WEIGHT * s["cf"] +
                      self.PREF_WEIGHT * s["pref"])
            reason = self._explain(s)
            final.append((self.startup_map[sid], score, reason))

        final.sort(key=lambda x: x[1], reverse=True)
        return final[:n]

    def get_feed(self, investor_id: str, n: int = 20) -> List[Tuple[Startup, float, str]]:
        history = list(self.tracker.history[investor_id])
        if not history:
            ranked = sorted(self.startups, key=lambda s: s.traction + s.team_score, reverse=True)
            return [(s, (s.traction + s.team_score) / 20.0, "trending") for s in ranked[:n]]
        recent_id = history[-1].startup_id
        return self.recommend(investor_id, seed_startup_id=recent_id, n=n, exclude_seen=True)

    def _collaborative_scores(self, investor_id: str, candidates: List[str]) -> Dict[str, float]:
        all_ids    = list(self.investor_map.keys())
        inv_vecs   = np.array([
            self.tracker.interaction_vector(iid, self.startup_ids) for iid in all_ids
        ])
        target_vec = self.tracker.interaction_vector(investor_id, self.startup_ids)
        if target_vec.sum() == 0:
            return {}
        sims  = cosine_similarity(target_vec.reshape(1, -1), inv_vecs)[0]
        top_k = sorted(
            [(all_ids[i], float(s)) for i, s in enumerate(sims) if all_ids[i] != investor_id],
            key=lambda x: x[1], reverse=True
        )[:5]
        cf        = defaultdict(float)
        total_sim = sum(s for _, s in top_k) or 1.0
        for iid, sim in top_k:
            vec = self.tracker.interaction_vector(iid, self.startup_ids)
            for i, sid in enumerate(self.startup_ids):
                if sid in candidates:
                    cf[sid] += sim * vec[i] / total_sim
        if cf:
            max_cf = max(cf.values()) or 1.0
            cf     = {k: v / max_cf for k, v in cf.items()}
        return dict(cf)

    @staticmethod
    def _explain(s: Dict[str, float]) -> str:
        dominant = max(s, key=s.get)
        return {
            "cb":   "similar to startups you viewed",
            "cf":   "investors like you funded this",
            "pref": "matches your sector preference",
        }[dominant]

print("recommendation engine defined")

recommendation engine defined


## 9. initialise engine

In [11]:
engine = RecommendationEngine(STARTUPS, INVESTORS)

building feature matrix...
engine ready — 45 startups, 10 investors


## 10. cold start feed

In [12]:
investor_id = "inv_001"
investor    = engine.investor_map[investor_id]

print(f"investor : {investor.name} ({investor.firm})")
print(f"sectors  : {', '.join(investor.preferred_sectors)}")
print(f"stages   : {', '.join(investor.preferred_stages)}")
print(f"focus    : {investor.focus}")
print()

feed = engine.get_feed(investor_id, n=6)

print(f"cold start feed — sorted by traction + team score (no history yet):")
print()
print(f"{'rank':<5} {'name':<18} {'sector':<14} {'stage':<12} {'arr ($m)':>8}  {'reason'}")
print("-" * 72)
for rank, (s, score, reason) in enumerate(feed, 1):
    print(f"{rank:<5} {s.name:<18} {s.sector:<14} {s.stage:<12} {s.revenue_arr:>8.1f}  {reason}")

investor : Ananya Kapoor (Inflection Ventures)
sectors  : FinTech, SaaS, DeepTech
stages   : Seed, Series A
focus    : B2B enterprise software and financial infrastructure for emerging markets

cold start feed — sorted by traction + team score (no history yet):

rank  name               sector         stage        arr ($m)  reason
------------------------------------------------------------------------
1     WindCell           CleanTech      Series B         12.4  trending
2     CellCraft          BioTech        Series B         13.5  trending
3     InsureIQ           FinTech        Series B          7.8  trending
4     SurgAI             HealthTech     Series B          9.2  trending
5     BioFab             DeepTech       Series B         10.8  trending
6     MediScan AI        HealthTech     Series A          3.1  trending


## 11. record investor interactions

In [13]:
primary_sector   = investor.preferred_sectors[0]
secondary_sector = investor.preferred_sectors[1]

primary_picks   = [s for s in STARTUPS if s.sector == primary_sector][:3]
secondary_picks = [s for s in STARTUPS if s.sector == secondary_sector][:2]

for s in primary_picks:
    engine.record_interaction(investor_id, s.id, "click")

engine.record_interaction(investor_id, primary_picks[0].id, "save")

for s in secondary_picks:
    engine.record_interaction(investor_id, s.id, "click")

print(f"clicks recorded on {primary_sector}   : {[s.name for s in primary_picks]}")
print(f"save   recorded                       : {primary_picks[0].name}")
print(f"clicks recorded on {secondary_sector} : {[s.name for s in secondary_picks]}")

clicks recorded on FinTech   : ['PayNova', 'WealthPath', 'InsureIQ']
save   recorded                       : PayNova
clicks recorded on SaaS : ['CloudStack Pro', 'HRMagic']


## 12. sector affinity profile

In [14]:
sector_weights = engine.tracker.get_sector_weights(investor_id)
sorted_sectors = sorted(sector_weights.items(), key=lambda x: x[1], reverse=True)

print(f"sector affinity profile for {investor.name}:")
print()
for sector, w in sorted_sectors:
    bar = "#" * int(w * 40)
    print(f"  {sector:<15} {bar:<42} {w:.3f}")

sector affinity profile for Ananya Kapoor:

  FinTech         ##############################             0.750
  SaaS            ##########                                 0.250


## 13. personalised recommendations

In [15]:
seed = primary_picks[-1]
recs = engine.recommend(investor_id, seed_startup_id=seed.id, n=8)

print(f"recommendations seeded from : {seed.name} ({seed.sector})")
print()
print(f"{'rank':<5} {'name':<18} {'sector':<14} {'stage':<12} {'score':>6}  {'reason'}")
print("-" * 78)
for rank, (s, score, reason) in enumerate(recs, 1):
    print(f"{rank:<5} {s.name:<18} {s.sector:<14} {s.stage:<12} {score:>6.3f}  {reason}")

recommendations seeded from : InsureIQ (FinTech)

rank  name               sector         stage         score  reason
------------------------------------------------------------------------------
1     TradeNest          FinTech        Series A      0.466  matches your sector preference
2     SupplySync         SaaS           Series B      0.370  similar to startups you viewed
3     LoanDost           FinTech        Pre-Seed      0.346  matches your sector preference
4     BioFab             DeepTech       Series B      0.317  similar to startups you viewed
5     GrocerHub          E-Commerce     Series B      0.300  similar to startups you viewed
6     LangLeap           EdTech         Series B      0.295  similar to startups you viewed
7     WindCell           CleanTech      Series B      0.293  similar to startups you viewed
8     CellCraft          BioTech        Series B      0.288  similar to startups you viewed


## 14. startup detail view

In [16]:
top = recs[0][0]
print(f"startup detail — {top.name}")
print()
print(f"  sector        : {top.sector}")
print(f"  stage         : {top.stage}")
print(f"  location      : {top.location}")
print(f"  founded       : {top.founded_year}")
print(f"  employees     : {top.employees}")
print(f"  arr           : ${top.revenue_arr}m")
print(f"  funding ask   : ${top.funding_needed}m")
print(f"  market size   : ${top.market_size}b")
print(f"  traction      : {top.traction}/10")
print(f"  team score    : {top.team_score}/10")
print(f"  business model: {top.business_model}")
print(f"  website       : {top.website}")
print()
print(f"  problem")
print(f"  {top.problem}")
print()
print(f"  solution")
print(f"  {top.solution}")
print()
print(f"  description")
print(f"  {top.description}")
print()
print(f"  tags : {', '.join(top.tags)}")

startup detail — TradeNest

  sector        : FinTech
  stage         : Series A
  location      : Hyderabad, India
  founded       : 2021
  employees     : 54
  arr           : $1.9m
  funding ask   : $6.5m
  market size   : $150.0b
  traction      : 8.1/10
  team score    : 8.6/10
  business model: Per-trade fee + subscription
  website       : https://tradenest.in

  problem
  India's 35 million retail traders lose ₹2,200 crore annually due to emotional decision-making, lack of real-time risk alerts, and fragmented broker platforms that don't provide AI-driven insights.

  solution
  TradeNest is a broker-agnostic trading co-pilot that connects via broker APIs to provide real-time portfolio risk scores, sentiment-based entry/exit alerts derived from 800+ news sources, and an AI journaling feature that identifies behavioural trading patterns.

  description
  TradeNest has 92,000 active trader accounts with a 67% 90-day retention rate. Users who follow AI alerts show a statistically 

## 15. fund event — strongest signal

In [17]:
funded = recs[0][0]
engine.record_interaction(investor_id, funded.id, "fund")

print(f"funded : {funded.name} ({funded.sector})")
print("signal weight : 5.0x  — strongest possible signal")
print("all similar startups now receive a significant boost")
print()

post_fund = engine.recommend(investor_id, seed_startup_id=funded.id, n=6)

print("updated recommendations after fund event:")
print()
print(f"{'rank':<5} {'name':<18} {'sector':<14} {'stage':<12} {'score':>6}  {'reason'}")
print("-" * 78)
for rank, (s, score, reason) in enumerate(post_fund, 1):
    print(f"{rank:<5} {s.name:<18} {s.sector:<14} {s.stage:<12} {score:>6.3f}  {reason}")

funded : TradeNest (FinTech)
signal weight : 5.0x  — strongest possible signal
all similar startups now receive a significant boost

updated recommendations after fund event:

rank  name               sector         stage         score  reason
------------------------------------------------------------------------------
1     LoanDost           FinTech        Pre-Seed      0.379  matches your sector preference
2     LegalEase          SaaS           Series A      0.290  similar to startups you viewed
3     NeuroChip          DeepTech       Series A      0.288  similar to startups you viewed
4     SkillForge         EdTech         Series A      0.275  similar to startups you viewed
5     MediScan AI        HealthTech     Series A      0.269  similar to startups you viewed
6     RoboSort           DeepTech       Series A      0.268  similar to startups you viewed


## 16. second investor session — different preferences

In [18]:
inv_b      = "inv_003"
investor_b = engine.investor_map[inv_b]

print(f"investor : {investor_b.name} ({investor_b.firm})")
print(f"sectors  : {', '.join(investor_b.preferred_sectors)}")
print(f"focus    : {investor_b.focus}")
print()

agri_picks  = [s for s in STARTUPS if s.sector == "AgriTech"][:3]
clean_picks = [s for s in STARTUPS if s.sector == "CleanTech"][:2]

for s in agri_picks:
    engine.record_interaction(inv_b, s.id, "click")
engine.record_interaction(inv_b, agri_picks[0].id, "save")
for s in clean_picks:
    engine.record_interaction(inv_b, s.id, "click")

sector_w_b = engine.tracker.get_sector_weights(inv_b)
print("sector affinity profile:")
print()
for sector, w in sorted(sector_w_b.items(), key=lambda x: x[1], reverse=True):
    bar = "#" * int(w * 40)
    print(f"  {sector:<15} {bar:<42} {w:.3f}")

print()
recs_b = engine.recommend(inv_b, seed_startup_id=agri_picks[-1].id, n=6)
print("recommendations for investor b:")
print()
print(f"{'rank':<5} {'name':<18} {'sector':<14} {'stage':<12} {'score':>6}  {'reason'}")
print("-" * 78)
for rank, (s, score, reason) in enumerate(recs_b, 1):
    print(f"{rank:<5} {s.name:<18} {s.sector:<14} {s.stage:<12} {score:>6.3f}  {reason}")

investor : Priya Nair (GreenFuture Fund)
sectors  : CleanTech, AgriTech, SaaS
focus    : Climate tech, sustainable agriculture, and circular economy

sector affinity profile:

  AgriTech        ##############################             0.750
  CleanTech       ##########                                 0.250

recommendations for investor b:

rank  name               sector         stage         score  reason
------------------------------------------------------------------------------
1     AquaNova           AgriTech       Seed          0.456  matches your sector preference
2     PoultryIQ          AgriTech       Pre-Seed      0.404  matches your sector preference
3     FreshRoute         E-Commerce     Series A      0.281  similar to startups you viewed
4     CloudStack Pro     SaaS           Series A      0.255  similar to startups you viewed
5     StyleMart          E-Commerce     Series A      0.254  similar to startups you viewed
6     WindCell           CleanTech      Series B 

## 17. personalisation proof — same startup, different scores per investor

In [19]:
target = engine.startup_map["startup_016"]

print(f"startup : {target.name} ({target.sector} | {target.stage})")
print(f"problem : {target.problem[:95]}...")
print()
print(f"{'investor':<22} {'firm':<26} {'primary sector':<15} {'score':>6}")
print("-" * 74)
for inv in INVESTORS:
    recs_inv = engine.recommend(inv.id, seed_startup_id=target.id, n=1)
    score    = recs_inv[0][1] if recs_inv else 0.0
    print(f"{inv.name:<22} {inv.firm:<26} {inv.preferred_sectors[0]:<15} {score:>6.3f}")

startup : SolarGrid (CleanTech | Series A)
problem : India's 30 million SME manufacturers pay 22-28% of revenues on electricity. Rooftop solar adopt...

investor               firm                       primary sector   score
--------------------------------------------------------------------------
Ananya Kapoor          Inflection Ventures        FinTech          0.309
Rajan Mehta            BioHorizon Capital         BioTech          0.304
Priya Nair             GreenFuture Fund           CleanTech        0.350
Vikram Sinha           GameOn Ventures            GameTech         0.284
Sneha Agarwal          EduQuest Capital           EdTech           0.286
Rahul Tiwari           DeepTech Foundry           DeepTech         0.295
Meera Joshi            AgriGrowth Partners        AgriTech         0.287
Arjun Bose             ConsumerFirst Capital      E-Commerce       0.284
Deepika Rao            HealthVentures India       HealthTech       0.304
Suresh Kumar           Impact Alpha Fund  

## 18. sector distribution in dataset

In [20]:
sector_counts = startups_df["sector"].value_counts()
stage_counts  = startups_df["stage"].value_counts()

print("startups by sector:")
for sector, count in sector_counts.items():
    bar = "#" * count
    print(f"  {sector:<15} {bar:<20} {count}")

print()
print("startups by stage:")
for stage, count in stage_counts.items():
    bar = "#" * count
    print(f"  {stage:<12} {bar:<20} {count}")

startups by sector:
  FinTech         #####                5
  HealthTech      #####                5
  EdTech          #####                5
  CleanTech       #####                5
  AgriTech        #####                5
  DeepTech        #####                5
  SaaS            #####                5
  E-Commerce      #####                5
  BioTech         ###                  3
  GameTech        ##                   2

startups by stage:
  Seed         #################    17
  Series A     ###############      15
  Series B     ########             8
  Pre-Seed     #####                5


## 19. top startups by traction and team score

In [21]:
startups_df["composite_score"] = startups_df["traction"] + startups_df["team_score"]
top10 = startups_df.nlargest(10, "composite_score")[
    ["name", "sector", "stage", "traction", "team_score", "composite_score", "revenue_arr"]
].reset_index(drop=True)

print("top 10 startups by traction + team score:")
print()
print(top10.to_string(index=True))

top 10 startups by traction + team score:

          name      sector     stage  traction  team_score  composite_score  revenue_arr
0     WindCell   CleanTech  Series B       9.3         9.5             18.8         12.4
1    CellCraft     BioTech  Series B       9.0         9.8             18.8         13.5
2     InsureIQ     FinTech  Series B       9.2         9.4             18.6          7.8
3       SurgAI  HealthTech  Series B       9.0         9.6             18.6          9.2
4       BioFab    DeepTech  Series B       8.8         9.7             18.5         10.8
5  MediScan AI  HealthTech  Series A       8.9         9.3             18.2          3.1
6     LangLeap      EdTech  Series B       9.1         9.0             18.1          8.5
7   SupplySync        SaaS  Series B       9.0         9.1             18.1          8.6
8    GrocerHub  E-Commerce  Series B       9.1         8.9             18.0         11.2
9      PayNova     FinTech  Series A       8.7         9.1         